In [1]:
import pandas as pd

In [3]:
from pathlib import Path

In [24]:
import os


'Options Selling.ods'

In [50]:
# Load the data
# df = pd.read_csv("data.csv")
# df = pd.read_csv("../Option Selling.ods")
filename = os.listdir("..")[-3]
df = pd.read_excel(f"../{filename}", engine="odf", skiprows=40, sheet_name="Hourly_Futures")
print(df.columns.tolist())
# 1. Cleanup
print(f'lengthbefore {len(df)}')
df = df.dropna().query("`Entry Date` != 'Entry Date'").replace({'₹': '', ',': '','%':''}, regex=True).reset_index(drop=True)
print(f'lengthbefore {len(df)}')
# 2. Convert Numeric Columns (Floats & Integers)
numeric_cols = [
    'Entry', 'SL', 'Target', 'Spot points', 
    'Option Entry', 'Option Exit', 'Option pts', 
    'Lot', 'PnL', 'Return on Cap', 'Capital'
]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 3. Convert Date Columns
date_cols = ['Entry Date', 'Exit Date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%d/%m/%y', errors='coerce')

# 4. Convert Time Columns (Optional: keeps them as clean string or converts to time)
time_cols = ['Entry Time', 'Exit Time']
for col in time_cols:
    df[col] = pd.to_datetime(df[col], format='%H:%M:%S', errors='coerce').dt.time

['Entry Date', 'Entry Time', 'Entry', 'SL', 'Target', 'Exit Date', 'Exit Time', 'Entry TF', 'P or L', 'Spot points', 'Type', 'Option', 'Option Entry', 'Option Exit', 'Option pts', 'Expiry', 'Spot-option ratio', 'Unnamed: 17', 'Lot', 'PnL', 'Return on Cap', 'Capital']
lengthbefore 81
lengthbefore 43


/var/folders/9h/w9n0b4pn2vl1h92j3v3kfnz00000gn/T/ipykernel_9107/3125142414.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.dropna().query("`Entry Date` != 'Entry Date'").replace({'₹': '', ',': '','%':''}, regex=True).reset_index(drop=True)


In [53]:
df = df.rename(columns={"Unnamed: 17": "Moneyness"})
df.columns

Index(['Entry Date', 'Entry Time', 'Entry', 'SL', 'Target', 'Exit Date',
       'Exit Time', 'Entry TF', 'P or L', 'Spot points', 'Type', 'Option',
       'Option Entry', 'Option Exit', 'Option pts', 'Expiry',
       'Spot-option ratio', 'Moneyness', 'Lot', 'PnL', 'Return on Cap',
       'Capital'],
      dtype='object')

<h1>Plots</h1>

In [54]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Choose one depending on your environment:
pio.renderers.default = "browser"   # Opens chart in a new browser tab
# OR
# pio.renderers.default = "notebook"  # Forces basic notebook renderer

In [42]:
# Ensure df is sorted by date
df = df.sort_values('Exit Date').reset_index(drop=True)

# 1. Monthly PnL
df['Year-Month'] = df['Exit Date'].dt.to_period('M').astype(str)
monthly_pnl = df.groupby('Year-Month')['PnL'].sum().reset_index()
monthly_pnl['Status'] = monthly_pnl['PnL'].apply(lambda x: 'Profit' if x >= 0 else 'Loss')

In [45]:
df.head()

,Entry Date,Entry Time,Entry,SL,Target,Exit Date,Exit Time,Entry TF,P or L,Spot points,...,Option Exit,Option pts,Expiry,Spot-option ratio,Moneyness,Lot,PnL,Return on Cap,Capital,Year-Month
0,2026-01-12,14:15:00,59395,59045,59995,2026-01-16,09:15:00,1h,P,600,...,1200,245,Jan,0.408333,ITM,60,14700.0,0.055472,291700.0,2026-01
1,2026-01-16,12:15:00,59980,60165,59610,2026-01-16,15:15:00,1h,L,-185,...,790,-110,Feb,0.594595,OTM,60,-6600.0,-0.024906,285100.0,2026-01
2,2026-01-19,13:15:00,59920,60240,59410,2026-01-20,15:15:00,1h,P,510,...,1000,200,Feb,0.392157,OTM,60,12000.0,0.045283,297100.0,2026-01
3,2026-01-21,12:15:00,59020,58670,59620,2026-01-23,13:15:00,1h,L,-350,...,840,-160,Feb,0.457143,OTM,60,-9600.0,-0.036226,287500.0,2026-01
4,2026-01-28,12:15:00,59512,59712,58862,2026-01-29,10:15:00,1h,L,-200,...,790,-90,Feb,0.450000,OTM,60,-5400.0,-0.020377,282100.0,2026-01


In [44]:
fig_monthly = px.bar(
    monthly_pnl,
    x='Year-Month',
    y='PnL',
    color='Status',
    color_discrete_map={'Profit': '#26a69a', 'Loss': '#ef5350'},
    text_auto='.2f',
    title='Monthly Profit & Loss'
)
fig_monthly.update_layout(
    yaxis_title="PnL (₹)",
    xaxis_title="Month",
    showlegend=False
)
fig_monthly.show()

<h3>Capital Growth</h3>

In [118]:
# 2. Capital Growth
initial_capital = 110000
df['Cumulative_PnL'] = df['PnL'].cumsum()
df['Total_Capital'] = initial_capital + df['Cumulative_PnL']

In [96]:
fig_capital = px.line(
    df,
    x='Exit Date',
    y='Total_Capital',
    title='Capital Growth Curve',
    markers=True
)
# Add initial capital reference line
fig_capital.add_hline(
    y=initial_capital, 
    line_dash="dash", 
    line_color="gray", 
    annotation_text="Starting Capital (₹1,10,000)"
)
fig_capital.update_traces(line_color='#2962ff', line_width=2.5)
fig_capital.update_layout(yaxis_title="Total Capital (₹)", xaxis_title="Date")
fig_capital.show()

In [119]:
# Create the Capital Growth Line Plot
fig = px.line(
    df,
    x='Exit Date',
    y='Capital',
    title='Capital Growth Curve',
    markers=True,
    # Pass additional columns to hover_data so they can be formatted easily
    custom_data=['Capital', 'Return on Cap']
)

# Customize the hover tooltip to show both values clearly
fig.update_traces(
    hovertemplate=(
        "<b>Date:</b> %{x|%Y-%m-%d}<br>" +
        "<b>Capital:</b> ₹%{customdata[0]:,.2f}<br>" +
        "<b>Return on Capital:</b> %{customdata[1]:.2f}%" +
        "<extra></extra>"  # Removes default trace name box
    ),
    line_color='#2962ff',
    line_width=2.5
)

# Optional: Add a reference horizontal line for initial capital if needed
initial_capital = df['Capital'].iloc[0]
fig.add_hline(
    y=initial_capital,
    line_dash="dash",
    line_color="gray",
    annotation_text="Starting Capital"
)

# Customize Layout
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Capital (₹)",
    hovermode="x unified"
)

fig.show()

<h3>PnL Depiction in Pie Chart with Average RR</h3>

In [121]:
# 3. Win / Loss Counts
df['Result'] = df['PnL'].apply(lambda x: 'Win' if x > 0 else ('Loss' if x < 0 else 'Breakeven'))
win_loss_counts = df['Result'].value_counts().reset_index()
win_loss_counts.columns = ['Result', 'Count']

<h4>RR within pie chart</h4>

In [4]:
import numpy as np

# Calculate x = abs(Target - Entry) / abs(Entry - SL)
df['RR_Ratio'] = (df['Target'] - df['Entry']).abs() / (df['Entry'] - df['SL']).abs()

# Replace infinite values (in case SL == Entry by mistake) with NaN
df['RR_Ratio'] = df['RR_Ratio'].replace([np.inf, -np.inf], np.nan)

# Compute average Risk-to-Reward across all valid trades
avg_rr = df['RR_Ratio'].mean()

In [127]:
fig_winloss = px.pie(
    win_loss_counts,
    names='Result',
    values='Count',
    hole=0.5,
    title='Win / Loss Ratio & Avg R:R',
    color='Result',
    color_discrete_map={'Win': '#26a69a', 'Loss': '#ef5350', 'Breakeven': '#78909c'}
)

# Text inside slices formatted to white
fig_winloss.update_traces(
    textinfo='percent+label+value',
    insidetextfont=dict(color='white', size=14)
)

# Add 1:x inside the center hole
fig_winloss.add_annotation(
    text=f"<b>Avg R:R</b><br>1 : {avg_rr:.2f}",
    x=0.5,
    y=0.5,
    font=dict(size=16, color="black"),  # Use "white" if working in dark mode
    showarrow=False
)

fig_winloss.show()

<h3>Timeframe analytics | Timeframe Distribution & Win vs Loss per Timeframe</h3>

In [10]:
# 1. Total counts per Timeframe (for Pie chart)
tf_counts = df['Entry TF'].value_counts().reset_index()
tf_counts.columns = ['TF', 'Count']

# 2. Wins vs Losses per Timeframe (for Bar chart)
tf_pl_counts = df.groupby(['Entry TF', 'P or L']).size().unstack(fill_value=0)

In [20]:
# Create 1 row, 2 columns subplot layout
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}]],  # domain for pie, xy for bar chart
    subplot_titles=("Timeframe Distribution", "Win vs Loss per Timeframe")
)

# --- 1. Left Chart: Timeframe Pie Chart ---
fig.add_trace(
    go.Pie(
        labels=tf_counts['TF'],
        values=tf_counts['Count'],
        hole=0.4,
        textinfo='percent+label+value',
        insidetextfont=dict(color='white', size=14),
        marker=dict(colors=['#5c6bc0', '#26a69a'])
    ),
    row=1, col=1
)

# --- 2. Right Chart: Win vs Loss Bar Chart ---
# Add Wins ('P') Bar
if 'P' in tf_pl_counts.columns:
    fig.add_trace(
        go.Bar(
            x=tf_pl_counts.index,
            y=tf_pl_counts['P'],
            name='Profit (P)',
            marker_color='#26a69a',
            text=tf_pl_counts['P'],
            textposition='auto'
        ),
        row=1, col=2
    )

# Add Losses ('L') Bar
if 'L' in tf_pl_counts.columns:
    fig.add_trace(
        go.Bar(
            x=tf_pl_counts.index,
            y=tf_pl_counts['L'],
            name='Loss (L)',
            marker_color='#ef5350',
            text=tf_pl_counts['L'],
            textposition='auto'
        ),
        row=1, col=2
    )

# --- Layout Configuration ---
fig.update_layout(
    title_text="Timeframe Analytics Dashboard",
    barmode='group',  # Displays 'P' and 'L' side-by-side for 15m and 1h
    height=500,
    showlegend=False,
    xaxis_title="Timeframe",
    yaxis_title="Number of Trades"
)

fig.show()